# 🧠 EMS741 Exam Prep: Ollama & Study Environment
This notebook installs Ollama natively into your Conda/Mamba environment (with GPU acceleration), starts the server, and provides a structured workspace for your bookclub-style exam focusing on Lagergren (2020) and Li (2020).

In [8]:
# 1. Install dependencies natively via Conda/Pip (Forcing CUDA/GPU variant)
!mamba install -y -c conda-forge ollama "ollama=*=*cuda*"
!pip install ollama -q

import subprocess
import time
import ollama

# 2. Start the background server
print("Starting GPU-enabled Ollama server...")
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 3. Pull the model
MODEL_NAME = 'llama3'
print(f"Pulling {MODEL_NAME}...")
!ollama pull {MODEL_NAME}
print("✅ Setup Complete!")


Looking for: ['ollama', 'ollama=[build=*cuda*]']

[+] 0.0s
conda-forge/linux-64                                          No change
[+] 0.1s
conda-forge/noarch ━━━━━━━━━━━╸━━━━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.1s[+] 0.2s
conda-forge/noarch ━━━━━━━━━━━━━╸━━━━━━━━━━━  14.8MB /  26.2MB @  80.4MB/s  0.2s[+] 0.3s
conda-forge/noarch ━━━━━━━━━━━━━━━━━━━╸━━━━━  21.3MB /  26.2MB @  90.6MB/s  0.3sconda-forge/noarch                                  26.2MB @  97.1MB/s  0.4s

Pinned packages:
  - python 3.11.*


Transaction

  Prefix: /home/jovyan/env/mamba-gpu

  All requested packages already installed

Starting GPU-enabled Ollama server...
Pulling llama3...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                   

In [2]:
def query_exam_tutor(prompt, system_context="You are an expert AI tutor helping a robotics MSc student prepare for their EMS741 exam."):
    """Streams the answer from Ollama using the official python client."""
    print(f"\n📝 Question: {prompt}\n")
    print("🤖 Answer:")
    
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': system_context},
        {'role': 'user', 'content': prompt}
    ], stream=True)
    
    for chunk in response:
        print(chunk['message']['content'], end='', flush=True)
    print("\n\n" + "━"*60)


Loading Context

In [9]:
def load_context(filepath):
    with open(filepath, 'r') as file:
        return file.read()

# Load the dense markdown file you created
my_custom_context = load_context('Ollama-Course-Context-Pack.md')

query_exam_tutor(
    "How does the constraint loss in BINNs enforce biological priors?", 
    system_context=my_custom_context
)


📝 Question: How does the constraint loss in BINNs enforce biological priors?

🤖 Answer:
According to the Lagergren MASTER CONTEXT, the Constraint Loss (Biological Priors) in BINNs (Biologically-informed neural networks) enforces biological priors by penalizing the network if:

1. Diffusivity (`D`) exits biological bounds `[D_min, D_max]`.
2. Growth (`G`) exits biological bounds `[G_min, G_max]`.
3. Diffusivity decreases with density (`\partial D/\partial u < 0`).
4. Growth increases with density (`\partial G/\partial u > 0`).

These penalties ensure that the learned PDE functions are physically plausible and adhere to known biological constraints. This is in contrast to traditional PINNs, which do not explicitly enforce such priors.

By incorporating these biological constraints into the loss function, BINNs encourage the network to learn a more biologically informed representation of the reaction-diffusion system, rather than simply fitting the noisy data.

━━━━━━━━━━━━━━━━━━━━━━━━━━

## 🧪 Paper 1: Lagergren 2020 (BINNs)
**Core concepts:** Physics-informed loss, learning unknown PDE terms (diffusivity/growth), scratch-assays, time-delay term, constraints/priors.

In [10]:
ctx_lagergren = """
Context: Lagergren 2020 'Biologically-informed neural networks'.
BINNs learn unknown reaction-diffusion PDE terms directly from sparse scratch-assay data. 
They use an MLP surrogate for density u(x,t), and MLPs for parameters D(u) and G(u). 
They added a time-delay term T(t) to handle initial confluency discrepancy.
"""

query_exam_tutor(
    "Describe the main building blocks of the BINN model (inputs, outputs, architecture, loss function).", 
    system_context=ctx_lagergren
)


📝 Question: Describe the main building blocks of the BINN model (inputs, outputs, architecture, loss function).

🤖 Answer:
The Biologically-informed Neural Networks (BINNs) model consists of the following main building blocks:

**Inputs:**

* Scratch-assay data: sparse measurements of cell density u(x,t) at different spatial locations x and times t
* Spatial coordinates x
* Time t

**Outputs:**

* Density u(x,t): an estimate of the cell density at each spatial location x and time t
* Parameters D(u): an estimate of the reaction term coefficients
* G(u): an estimate of the diffusion term coefficients
* T(t): an estimate of the time-delay term to handle initial confluency discrepancy

**Architecture:**

1. **MLP Surrogate**: A multi-layer perceptron (MLP) is used as a surrogate model to approximate the density u(x,t). The MLP takes the spatial coordinates x and time t as inputs and outputs an estimate of the cell density.
2. **Parameter Networks**: Two separate MLPs are used to learn th

## 🧪 Paper 2: Li 2020 (CNN Surrogate)
**Core concepts:** Encoder-decoder CNN, 2D reaction-diffusion, purely data-driven (trained on 1M FEM samples), no physics in loss, ~300x speedup.

In [11]:
ctx_li = """
Context: Li 2020 'CNN surrogate for reaction-diffusion'.
A CNN encoder-decoder predicts 2D concentration fields bypassing FEM.
Inputs: 4-channel tensor (geometry/BC, D, K, time) as 21x21 matrices.
Data-driven (MSE loss on synthetic FEM data), no physical constraints in loss.
"""

query_exam_tutor(
    "What are the inputs, outputs, and architecture of this CNN model? How does it handle time-dependence?", 
    system_context=ctx_li
)


📝 Question: What are the inputs, outputs, and architecture of this CNN model? How does it handle time-dependence?

🤖 Answer:
Based on the context, here's what I understand about the CNN model:

**Inputs:**

The input to the CNN is a 4-channel tensor represented as a 21x21 matrix. The channels are:

1. Geometry/BC (Boundary Condition)
2. Diffusion coefficient (D)
3. Reaction rate constant (K)
4. Time

These inputs seem to be spatially-varying fields, with time being an additional dimension.

**Architecture:**

The CNN model appears to have an encoder-decoder architecture, which is commonly used in image-to-image translation tasks. The encoder takes the input tensor and produces a latent representation, which is then decoded into a predicted output.

**Outputs:**

The output of the CNN is a 2D concentration field, which is also represented as a 21x21 matrix. This output corresponds to the predicted spatial distribution of a chemical species at a given time point.

**Time-Dependence Hand

## ⚖️ Cross-Paper Comparison
**Core concepts:** Physics-informed (BINNs) vs Physics-based training data (Li CNN).

In [12]:
ctx_compare = """
You are comparing BINNs (Lagergren, 2020) and CNN surrogates (Li, 2020).
Lagergren: Physics in the loss function, continuous MLP, sparse real data.
Li: Physics only in the training data generation, discrete 2D CNN, massive synthetic data.
"""

query_exam_tutor(
    "Compare how these two papers incorporate physical laws. Which one enforces physical constraints during training, and how?", 
    system_context=ctx_compare
)


📝 Question: Compare how these two papers incorporate physical laws. Which one enforces physical constraints during training, and how?

🤖 Answer:
The first paper by Lagergren (2020) incorporates physical laws into the loss function of a continuous Multi-Layer Perceptron (MLP). This is done by using a combination of reconstruction loss and a term that penalizes predictions that violate physical laws. Specifically, they use a physics-informed neural network (PINN) formulation, which involves minimizing the difference between the predicted output and the true output, as well as the difference between the predicted gradient and the exact gradient computed from the physical laws.

In contrast, the second paper by Li (2020) incorporates physical laws only during the training data generation process. They use a discrete 2D Convolutional Neural Network (CNN) to model the underlying physics, but do not enforce physical constraints during the training process itself. Instead, they generate massi

In [14]:
query_exam_tutor(
    "How does the constraint loss in BINNs enforce biological priors? Mention the specific mathematical constraints on Diffusivity.",
    system_context=my_custom_context
)


📝 Question: How does the constraint loss in BINNs enforce biological priors? Mention the specific mathematical constraints on Diffusivity.

🤖 Answer:
According to the Lagergren (2020) paper, "Biologically-informed neural networks" (BINNs), the Constraint Loss term (`L_Constr`) is designed to enforce biological priors that ensure the learned PDE functions remain physically plausible.

In particular, the constraint loss term penalizes the network if:

1. **Diffusivity does not decrease with density**: The mathematical constraint on Diffusivity (`D`): `∂D/∂u < 0`, where `u` is cell density.
2. **Growth rate does not increase with density**: The mathematical constraint on Growth Rate (`G`): `∂G/∂u > 0`.

By incorporating these constraints into the loss function, BINNs ensure that the learned PDE functions remain consistent with biological expectations. This helps to mitigate the issue of the network learning unrealistic or implausible solutions.

In other words, the constraint loss term e

In [15]:
query_exam_tutor(
    "Compare the loss functions and training data used in Lagergren and Li. How do these choices reflect the concepts of 'Physics-informed' vs 'Physics-based data'?",
    system_context=my_custom_context
)


📝 Question: Compare the loss functions and training data used in Lagergren and Li. How do these choices reflect the concepts of 'Physics-informed' vs 'Physics-based data'?

🤖 Answer:
Let's dive into the comparison!

**Lagergren (2020)**

* Loss Function: L_Total = L_GLS + L_PDE + L_Constr
	+ L_GLS: Generalized Least Squares (GLS) for observation noise
	+ L_PDE: Physics loss, enforcing the reaction-diffusion PDE residual via Automatic Differentiation
	+ L_Constr: Constraint loss, penalizing D and G if they exit biological bounds or violate monotonicity
* Training Data: Sparse, noisy scratch-assay data

**Li (2020)**

* Loss Function: Mean Squared Error (MSE) between the CNN prediction and FEM ground truth
* Training Data: Massive synthetic datasets generated by Finite Element Method (FEM)

Now, let's connect these choices to the concepts of 'Physics-informed' vs 'Physics-based data':

**Physics-informed (Lagergren)**

* The loss function explicitly incorporates the physical PDE residua